# Discounted Cash Flow Model

This notebook gives a compact, readable walkthrough of the DCF model used in the site. It is meant to be easy to inspect on GitHub and easy to run locally.

We use the same core Python model files that power the app, so the formulas and outputs stay aligned.

## What This Notebook Covers

- default model inputs
- forecast build
- EV to equity bridge
- sensitivity table
- a few plain-English notes on what the outputs mean

## Core Formula Logic

The implemented workflow follows the standard FCFF-style DCF:

- Revenue$_t$ = Revenue$_{t-1}$ × (1 + g$_t$)
- EBITDA$_t$ = Revenue$_t$ × EBITDA margin$_t$
- D&A$_t$ = Revenue$_t$ × d$_t$
- EBIT$_t$ = EBITDA$_t$ − D&A$_t$
- NOPAT$_t$ = EBIT$_t$ × (1 − t$_t$)
- NWC$_t$ = Revenue$_t$ × n$_t$
- ΔNWC$_t$ = NWC$_t$ − NWC$_{t-1}$
- CapEx$_t$ = Revenue$_t$ × c$_t$
- UFCF$_t$ = NOPAT$_t$ + D&A$_t$ − CapEx$_t$ − ΔNWC$_t$
- PV(UFCF$_t$) = UFCF$_t$ / (1 + WACC)$^t$
- TV$_n$ = UFCF$_n$ × (1 + g) / (WACC − g)

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Add the repository root so the shared python_model package resolves cleanly.
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from python_model.data_api import default_model_inputs
from python_model.dcf_engine import run_valuation


In [ ]:
# Start from the built-in example inputs so the notebook runs without any API dependency.
inputs = default_model_inputs()
valuation = run_valuation(inputs)

summary = valuation["summary"]
forecast_df = valuation["forecast_df"].copy()
bridge_df = valuation["bridge_df"].copy()
sensitivity_df = valuation["sensitivity_df"].copy()
diagnostics = valuation["diagnostics"]


## Summary Output

In [ ]:
summary_table = pd.DataFrame(
    [
        {"Metric": "Company", "Value": summary["company_name"]},
        {"Metric": "Ticker", "Value": summary["ticker"]},
        {"Metric": "Current Price", "Value": summary["current_price"]},
        {"Metric": "Computed WACC", "Value": summary["computed_wacc"]},
        {"Metric": "Terminal Growth", "Value": summary["terminal_growth_rate"]},
        {"Metric": "Enterprise Value", "Value": summary["enterprise_value"]},
        {"Metric": "Equity Value", "Value": summary["equity_value"]},
        {"Metric": "Implied Share Price", "Value": summary["implied_share_price"]},
        {"Metric": "Upside / Downside", "Value": summary["upside_downside"]},
        {"Metric": "Terminal Value Weight", "Value": summary["terminal_value_weight"]},
    ]
)
summary_table

## Forecast Table

This is the operating forecast that builds the valuation one year at a time.

In [ ]:
forecast_df

## EV to Equity Bridge

This bridge shows how we move from the value of operations to the value available to equity holders.

In [ ]:
bridge_df

## Sensitivity Table

This matrix shows how the implied share price moves when WACC and terminal growth assumptions change.

In [ ]:
sensitivity_df

## Diagnostics

These are quick warnings or interpretation cues surfaced by the model.

In [ ]:
pd.DataFrame({"Diagnostics": diagnostics})

## Next Step

From here, we can extend the notebook with:

- custom assumptions examples
- charts for revenue, EBITDA, and UFCF
- a worked example for a public company using fetched market data
- side-by-side comparison with the web app output